In [3]:
!nvidia-smi

Mon Sep 28 09:25:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 83.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (xformers)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dqqht7wo/unsloth_23032a9589494476887c0ce6c5f1ec6c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dqqht7wo/unsloth_23032a9589494476887c0ce6c5f1e

In [5]:
from unsloth import FastLanguageModel
import torch

# Set maximum sequence length (sufficient for contract clauses and JSON output)
max_seq_length = 2048  # Auto-detect precision (Float16 on Colab T4)
dtype = None  # Enable 4-bit quantization to fit within free GPU memory limits
load_in_4bit = True # Load pre-trained quantized base model and its tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [6]:
# Configure Parameter-Efficient Fine-Tuning (PEFT) using LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank dimension for low-rank adaptation matrices
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Apply LoRA to all attention and feed-forward linear layers
    lora_alpha = 16, # Scaling factor for LoRA weights
    lora_dropout = 0, # Optimized to 0 for maximum training speed in Unsloth
    bias = "none", # Keep bias frozen
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM using Unsloth checkpoints
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.9.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
from datasets import Dataset

# Define system instruction for the LLM
SYSTEM_PROMPT = """You are an expert German legal compliance and DSGVO/GDPR auditor.
Analyze the provided German contract clause or privacy statement.
Always return a valid JSON object with the following schema:
{
  "compliant": true/false,
  "risk_level": "None" / "Low" / "Medium" / "High",
  "gdpr_article": "Article reference or None",
  "issue": "Brief explanation in English",
  "recommendation": "Corrective action in English"
}"""

# Training dataset containing German enterprise clauses and auditor responses
training_data = [
    {
        "instruction": "Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.",
        "input": "Die personenbezogenen Daten der Nutzer werden auf unbestimmte Zeit gespeichert, um spätere geschäftliche Analysen ohne erneute Einwilligung zu ermöglichen.",
        "output": '{"compliant": false, "risk_level": "High", "gdpr_article": "Art. 5 Abs. 1 lit. e DSGVO (Speicherbegrenzung)", "issue": "Indefinite retention of personal data without defined deletion periods violates storage limitation principles.", "recommendation": "Define explicit retention periods and implement automated data erasure procedures."}'
    },
    {
        "instruction": "Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.",
        "input": "Zur Erbringung der Dienstleistungen übertragen wir Kundendaten an verbundene Subunternehmer in den USA, ohne dass zusätzliche Standardvertragsklauseln (SCC) abgeschlossen werden.",
        "output": '{"compliant": false, "risk_level": "High", "gdpr_article": "Art. 44 ff. DSGVO (Drittlandtransfer)", "issue": "International data transfer to a non-EU country without adequate safeguards (SCCs or adequacy decision) is illegal.", "recommendation": "Execute EU Standard Contractual Clauses (SCC) and perform a Transfer Impact Assessment (TIA)."}'
    },
    {
        "instruction": "Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.",
        "input": "Der Nutzer hat das Recht, jederzeit Auskunft über seine von uns verarbeiteten personenbezogenen Daten gemäß Art. 15 DSGVO zu verlangen sowie eine unentgeltliche Kopie zu erhalten.",
        "output": '{"compliant": true, "risk_level": "None", "gdpr_article": "Art. 15 DSGVO (Auskunftsrecht)", "issue": "None. The clause correctly states the data subject access right.", "recommendation": "Maintain operational readiness to fulfill requests within the statutory one-month timeframe."}'
    },
    {
        "instruction": "Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.",
        "input": "Mit der Unterzeichnung dieser Vereinbarung verzichtet der Betroffene unwiderruflich auf sein Recht, die Berichtigung oder Löschung seiner Daten zu verlangen.",
        "output": '{"compliant": false, "risk_level": "High", "gdpr_article": "Art. 16 & 17 DSGVO (Berichtigung und Löschung)", "issue": "Statutory data subject rights under GDPR cannot be waived by contract.", "recommendation": "Remove any waiver of mandatory rights; clauses waiving GDPR rights are legally void."}'
    },
    {
        "instruction": "Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.",
        "input": "Der Auftragsverarbeiter verarbeitet personenbezogene Daten ausschließlich auf dokumentierte Weisung des Verantwortlichen und stellt sicher, dass sich die zur Verarbeitung ermächtigten Personen zur Vertraulichkeit verpflichtet haben.",
        "output": '{"compliant": true, "risk_level": "None", "gdpr_article": "Art. 28 Abs. 3 DSGVO (Auftragsverarbeiter)", "issue": "None. Standard compliant Data Processing Agreement (DPA) clause.", "recommendation": "Ensure technical and organizational measures (TOMs) are documented and periodically reviewed."}'
    },
    {
        "instruction": "Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.",
        "input": "Im Falle einer Datenschutzverletzung wird das Unternehmen den Verantwortlichen innerhalb von 30 Werktagen informieren.",
        "output": '{"compliant": false, "risk_level": "Medium", "gdpr_article": "Art. 33 DSGVO (Meldung von Verletzungen)", "issue": "GDPR mandates reporting personal data breaches without undue delay and, where feasible, within 72 hours, not 30 business days.", "recommendation": "Adjust notification timeline to without undue delay and at the latest within 48 to 72 hours."}'
    }
]

# Function to format entries into Qwen2.5 ChatML prompt structure
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Format matching Qwen2.5 ChatML template
        text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{instruction}\n\nClause: {input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

# Convert dictionary list to Hugging Face Dataset format and apply template
raw_dataset = Dataset.from_list(training_data)
dataset = raw_dataset.map(formatting_prompts_func, batched = True)

# Inspect first formatted sample
print("Formatted sample preview:\n")
print(dataset["text"][0])

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Formatted sample preview:

<|im_start|>system
You are an expert German legal compliance and DSGVO/GDPR auditor.
Analyze the provided German contract clause or privacy statement.
Always return a valid JSON object with the following schema:
{
  "compliant": true/false,
  "risk_level": "None" / "Low" / "Medium" / "High",
  "gdpr_article": "Article reference or None",
  "issue": "Brief explanation in English",
  "recommendation": "Corrective action in English"
}<|im_end|>
<|im_start|>user
Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.

Clause: Die personenbezogenen Daten der Nutzer werden auf unbestimmte Zeit gespeichert, um spätere geschäftliche Analysen ohne erneute Einwilligung zu ermöglichen.<|im_end|>
<|im_start|>assistant
{"compliant": false, "risk_level": "High", "gdpr_article": "Art. 5 Abs. 1 lit. e DSGVO (Speicherbegrenzung)", "issue": "Indefinite retention of personal data without defined deletion periods violates storage limitation principles.", "recommendation": "De

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Configure Supervised Fine-Tuning (SFT) trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to False for short structured instruction tasks
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Total optimization steps
        learning_rate = 2e-4, # Optimal default for LoRA fine-tuning
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit", # Uses 8-bit AdamW optimizer to conserve VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Disable external tracking platforms like WandB
    ),
)

# Start training execution
trainer_stats = trainer.train()

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.022294
20,0.596259
30,0.073134
40,0.010239
50,0.008139
60,0.007779


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [9]:
# Enable native 2x faster inference engine in Unsloth
FastLanguageModel.for_inference(model)

# Define an unseen German contract clause for testing
test_clause = "Wir behalten uns das Recht vor, Kundendaten zu Werbezwecken an Dritte weiterzugeben, ohne den Nutzer vorher darüber zu informieren oder eine gesonderte Einwilligung einzuholen."

# Prepare input prompt using the exact ChatML template
prompt = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
Überprüfen Sie die folgende Klausel auf DSGVO-Konformität.

Clause: {test_clause}<|im_end|>
<|im_start|>assistant
"""

# Tokenize prompt and transfer tensors to GPU
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# Generate completion with greedy decoding
outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.1, # Low temperature ensures strict factual schema adherence
)

# Decode generated tokens back into readable text
decoded_output = tokenizer.batch_decode(outputs)
print(decoded_output[0].split("<|im_start|>assistant\n")[1].replace("<|im_end|>", ""))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"compliant": false, "risk_level": "High", "gdpr_article": "Art. 44 ff. DSGVO (Drittlandtransfer)", "issue": "International data transfer without adequate safeguards is illegal without prior adequacy decision or EU standard contract clauses.", "recommendation": "Execute EU Standard Contractual Clauses (SCC) and perform a Transfer Impact Assessment (TIA)."}


In [10]:
# Save fine-tuned LoRA adapters and tokenizer locally in the Colab environment
model.save_pretrained("dsgvo_qwen_lora")
tokenizer.save_pretrained("dsgvo_qwen_lora")

print("LoRA adapters and tokenizer saved successfully in './dsgvo_qwen_lora' directory.")

Unsloth: Restored added_tokens_decoder metadata in dsgvo_qwen_lora/tokenizer_config.json.


LoRA adapters and tokenizer saved successfully in './dsgvo_qwen_lora' directory.


In [11]:
# Convert and export merged model directly to 4-bit GGUF format for Ollama / llama.cpp
model.save_pretrained_gguf(
    "dsgvo_qwen_gguf",
    tokenizer,
    quantization_method = "q4_k_m" # Industry standard 4-bit quantization balance between speed and quality
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in dsgvo_qwen_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:58<00:00, 59.01s/it]


Unsloth: Merge process complete. Saved to `/content/dsgvo_qwen_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11160-mix-a6922cc (app-b11160-mix-a6922cc-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['dsgvo_qwen_gguf_gguf/Qwen2.5-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['dsgvo_qwen_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model dsgvo_qwen_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to dsgvo_qwen_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f dsgvo_qwen_gguf_gguf/Modelfile


{'save_directory': 'dsgvo_qwen_gguf',
 'gguf_directory': 'dsgvo_qwen_gguf_gguf',
 'gguf_files': ['dsgvo_qwen_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'dsgvo_qwen_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [12]:
# Verify generated GGUF file and inspect its file size
!ls -lh dsgvo_qwen_gguf

total 5.8G
-rw-r--r-- 1 root root 2.5K Sep 28 09:46 chat_template.jinja
-rw-r--r-- 1 root root 1.8K Sep 28 09:46 config.json
-rw-r--r-- 1 root root  239 Sep 28 09:46 generation_config.json
-rw-r--r-- 1 root root 3.7G Sep 28 09:49 model-00001-of-00002.safetensors
-rw-r--r-- 1 root root 2.1G Sep 28 09:50 model-00002-of-00002.safetensors
-rw-r--r-- 1 root root  35K Sep 28 09:46 model.safetensors.index.json
-rw-r--r-- 1 root root 6.9K Sep 28 09:46 tokenizer_config.json
-rw-r--r-- 1 root root  11M Sep 28 09:46 tokenizer.json


In [13]:
# Convert merged weights directory directly to 4-bit quantized GGUF format
model.save_pretrained_gguf(
    "dsgvo_qwen_q4",
    tokenizer,
    quantization_method = "q4_k_m" # Industry standard 4-bit quantization balance between speed and quality
)

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in dsgvo_qwen_q4/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:59<00:00, 59.96s/it]


Unsloth: Merge process complete. Saved to `/content/dsgvo_qwen_q4`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['dsgvo_qwen_q4_gguf/Qwen2.5-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['dsgvo_qwen_q4_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model dsgvo_qwen_q4_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to dsgvo_qwen_q4_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f dsgvo_qwen_q4_gguf/Modelfile


{'save_directory': 'dsgvo_qwen_q4',
 'gguf_directory': 'dsgvo_qwen_q4_gguf',
 'gguf_files': ['dsgvo_qwen_q4_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'dsgvo_qwen_q4_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [14]:
# Create Ollama Modelfile configuration for local GDPR compliance engine
modelfile_content = """FROM ./dsgvo_qwen_q4/unsloth.Q4_K_M.gguf

TEMPLATE \"\"\"<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
\"\"\"

SYSTEM \"\"\"You are an expert German legal compliance and DSGVO/GDPR auditor.
Analyze the provided German contract clause or privacy statement.
Always return a valid JSON object with the following schema:
{
  "compliant": true/false,
  "risk_level": "None" / "Low" / "Medium" / "High",
  "gdpr_article": "Article reference or None",
  "issue": "Brief explanation in English",
  "recommendation": "Corrective action in English"
}\"\"\"

PARAMETER temperature 0.1
PARAMETER stop <|im_end|>
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Ollama Modelfile generated successfully!")

Ollama Modelfile generated successfully!


In [17]:
# Install required decompression dependency
!apt-get update -qq && apt-get install -y zstd -qq

# Install Ollama binary inside the Colab environment
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama daemon in the background
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Allow daemon service to initialize

print("Ollama installed and daemon running successfully.")

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video grou

In [19]:
import os

# Find generated .gguf files inside current working directory
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".gguf"):
            print("Found GGUF file path:", os.path.join(root, file))

Found GGUF file path: ./dsgvo_qwen_q4_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf
Found GGUF file path: ./dsgvo_qwen_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf


In [20]:
# Write Modelfile pointing directly to the generated 4-bit GGUF binary
modelfile_content = """FROM ./dsgvo_qwen_q4_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf

TEMPLATE \"\"\"<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
\"\"\"

SYSTEM \"\"\"You are an expert German legal compliance and DSGVO/GDPR auditor.
Analyze the provided German contract clause or privacy statement.
Always return a valid JSON object with the following schema:
{
  "compliant": true/false,
  "risk_level": "None" / "Low" / "Medium" / "High",
  "gdpr_article": "Article reference or None",
  "issue": "Brief explanation in English",
  "recommendation": "Corrective action in English"
}\"\"\"

PARAMETER temperature 0.1
PARAMETER stop <|im_end|>
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile created successfully!")

Modelfile created successfully!


In [21]:
# Build Ollama local model image
!ollama create dsgvo-auditor -f Modelfile

In [22]:
# Run local inference query against the Ollama engine
!ollama run dsgvo-auditor "Klausel: Alle erfassten Mitarbeiterdaten werden ohne Verschlüsselung auf externen Servern gespeichert."

{"compliant": false, "risk_level": "High", "gdpr_article": "Art. 5 Abs. 4 D
DSGVO (Speicherbegrenzung)", "issue": "GDPR mandates encryption of personal
personal data stored on external servers to protect against data breaches."
breaches.", "recommendation": "Implement encryption of all stored data on e
external servers without compromising performance or security."}

